In [3]:
import sys
print(sys.executable)

/Users/Anneleonore/Documents/githubportfolio/delivery-agent/.venv/bin/python


In [5]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_parquet("../data/deliveries_clean.parquet")
df = df[~df["long_delivery"]].copy()

# Features known at the moment the order is placed
df["hour"] = df["created_local"].dt.hour
df["weekday"] = df["created_local"].dt.day_of_week
df["orders_per_dasher"] = df["total_outstanding_orders"] / df["total_onshift_dashers"].replace(0, np.nan)
df["busy_share"] = df["total_busy_dashers"] / df["total_onshift_dashers"].replace(0, np.nan)
df["store_primary_category"] = df["store_primary_category"].astype("category")

FEATURES = [
    "market_id", "store_primary_category", "order_protocol",
    "total_items", "num_distinct_items", "subtotal", "min_item_price", "max_item_price",
    "total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders",
    "orders_per_dasher", "busy_share",
    "estimated_order_place_duration", "estimated_store_to_consumer_driving_duration",
    "hour", "weekday",
]

# Split by time: the last 7 days are the test set (as if predicting next week)
cutoff = df["created_local"].max() - pd.Timedelta(days=7)
train, test = df[df["created_local"] < cutoff], df[df["created_local"] >= cutoff]
print(f"Train: {len(train):,} orders | Test: {len(test):,} orders (from {cutoff:%d %b})")

# Baseline: always predict the median from the training period
baseline = np.full(len(test), train["delivery_min"].median())

model = HistGradientBoostingRegressor(
    loss="absolute_error", max_iter=500, learning_rate=0.05,
    categorical_features="from_dtype", random_state=42,
)
model.fit(train[FEATURES], train["delivery_min"])
pred = model.predict(test[FEATURES])

def report(name, p):
    mae = mean_absolute_error(test["delivery_min"], p)
    within10 = (np.abs(test["delivery_min"] - p) <= 10).mean()
    print(f"{name:<10} average error: {mae:5.1f} min | within 10 min: {within10:.0%}")

report("Baseline", baseline)
report("Model", pred)

joblib.dump({"model": model, "features": FEATURES}, "../agent/model.joblib")
print("Model saved to agent/model.joblib")

Train: 144,044 orders | Test: 52,355 orders (from 10 Feb)
Baseline   average error:  13.6 min | within 10 min: 49%
Model      average error:  10.7 min | within 10 min: 61%
Model saved to agent/model.joblib


In [6]:
import sys
sys.path.insert(0, "../agent")
import tools

print(tools.run_sql("""
    SELECT weekday, round(median(delivery_min), 1) AS median_min, count(*) AS orders
    FROM deliveries GROUP BY weekday ORDER BY median_min DESC
"""))
print(tools.predict_delivery_time(market_id=2, store_category="pizza", hour=18, weekday="Friday"))
print(tools.predict_delivery_time(market_id=2, store_category="pizza", hour=18, weekday="Friday",
                                  total_onshift_dashers=20, total_outstanding_orders=60))

weekday,median_min,orders
Friday,48.0,32946
Saturday,45.3,32681
Sunday,44.9,30471
Monday,44.2,24426
Wednesday,43.7,25148
Thursday,41.8,26927
Tuesday,41.6,23800

Predicted delivery time: 50 minutes (typical error about 11 minutes). Inputs used: {'total_items': 3.0, 'subtotal': 22.0, 'total_onshift_dashers': 37.0, 'total_busy_dashers': 34.0, 'total_outstanding_orders': 41.0}
Predicted delivery time: 69 minutes (typical error about 11 minutes). Inputs used: {'total_items': 3.0, 'subtotal': 22.0, 'total_onshift_dashers': 20.0, 'total_busy_dashers': 34.0, 'total_outstanding_orders': 60.0}


In [1]:
import sys
sys.path.insert(0, "../agent")
import importlib, tools
importlib.reload(tools)

print(tools.TABLE_DESCRIPTION)
print(tools.predict_delivery_time(2, "pizza", 18, "Friday"))


Table `deliveries`: one row per DoorDash order. 196,399 orders from 6,740 stores,
placed between 21 January 2015 and 17 February 2015 (28 days).
Deliveries recorded as taking more than 5 hours are excluded as likely data errors.
Number of days per weekday in the data: Monday 4, Tuesday 4, Wednesday 4, Thursday 4, Friday 4, Saturday 4, Sunday 4.

Columns:
- market_id: anonymised region, 1 to 6 (all on the US West Coast; city names are not known)
- created_local: timestamp when the order was placed, in US Pacific time
- order_date (date), hour (0-23), weekday ('Monday' ... 'Sunday'): derived from created_local
- delivery_min: minutes from order to delivery (the outcome of interest)
- store_id; store_primary_category: cuisine, 75 values; most common:
  american, pizza, mexican, burger, sandwich, chinese, japanese, dessert, fast, indian, thai, italian, vietnamese, breakfast, mediterranean. 'unknown' means the category is missing.
- order_protocol: id of how the store receives orders; the 